# Web Search Tool with Strands Agent

## Overview

This notebook connects a Strands AI agent to the AgentCore Gateway created in notebook 01. The agent automatically discovers the Web Search tool via MCP `tools/list`, then invokes it when answering questions that require current information.

You will:
1. Connect to the Gateway using MCP Streamable HTTP transport
2. Discover the WebSearch tool at runtime
3. Create a Strands agent and ask real-time questions
4. Inspect the agent's tool calls and grounded responses

### Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Interactive (Jupyter Notebook) |
| AgentCore components | AgentCore Gateway |
| Agentic framework | Strands Agents |
| LLM model | Anthropic Claude Sonnet 4 (`us.anthropic.claude-sonnet-4-20250514-v1:0`) |
| Tutorial vertical | Cross-vertical |
| Example complexity | Easy |
| SDK used | boto3, strands-agents |

### How It Works

```
User question
      │
      ▼
Strands Agent (Claude Sonnet 4)
      │  decides to call WebSearch
      ▼
MCP tools/call  →  AgentCore Gateway  →  Web Search Backend
      │
      ▼
Structured results { text, url, title, publishedDate }
      │
      ▼
Final answer with cited sources
```

## Prerequisites

Complete notebook `01-web-search-gateway-setup.ipynb` first and set the following environment variables:

```bash
export AGENTCORE_GATEWAY_URL="..."
export COGNITO_DOMAIN="..."
export COGNITO_CLIENT_ID="..."
export COGNITO_CLIENT_SECRET="..."
export COGNITO_SCOPE="agentcore-websearch/invoke"
export AWS_DEFAULT_REGION="us-east-1"
```

### Required IAM permissions

~~~json
{
  "Effect": "Allow",
  "Action": "bedrock:InvokeModel",
  "Resource": "*"
}
~~~

## 1. Install Dependencies

In [ ]:
!pip install --upgrade -r requirements.txt --quiet

## 2. Configuration

In [ ]:
import os
import requests
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
GATEWAY_URL = os.environ["AGENTCORE_GATEWAY_URL"]
COGNITO_DOMAIN = os.environ["COGNITO_DOMAIN"]
CLIENT_ID = os.environ["COGNITO_CLIENT_ID"]
CLIENT_SECRET = os.environ["COGNITO_CLIENT_SECRET"]
SCOPE_STRING = os.environ["COGNITO_SCOPE"]
MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

print(f"Gateway URL : {GATEWAY_URL}")
print(f"Region      : {REGION}")
print(f"Model       : {MODEL_ID}")

## 3. Create OAuth Token and MCP Transport

In [ ]:
def get_token():
    """Retrieve a fresh OAuth access token from Cognito."""
    url = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"
    resp = requests.post(
        url,
        headers={"Content-Type": "application/x-www-form-urlencoded"},
        data={
            "grant_type": "client_credentials",
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "scope": SCOPE_STRING
        }
    )
    resp.raise_for_status()
    return resp.json()["access_token"]


def create_transport():
    """Create a fresh MCP Streamable HTTP transport with a valid token."""
    token = get_token()
    return streamablehttp_client(
        GATEWAY_URL,
        headers={"Authorization": f"Bearer {token}"}
    )


mcp_client = MCPClient(create_transport)
print("MCP client created ✓")

## 4. Discover Tools and Create the Agent

The agent discovers tools dynamically at runtime via `tools/list`. Strands registers them automatically — no manual tool definitions needed.

In [ ]:
SYSTEM_PROMPT = """You are a helpful research assistant with access to real-time web search.

PRINCIPLES:
- Use WebSearch to find current information before answering
- Always cite your sources with URLs
- Keep queries concise (under 200 characters)
- Synthesize information from multiple results when possible
"""

with mcp_client:
    tools = mcp_client.list_tools_sync()
    print(f"Discovered {len(tools)} tool(s): {[t.tool_name for t in tools]}")

    model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.7, max_tokens=1024)
    agent = Agent(model=model, tools=tools, system_prompt=SYSTEM_PROMPT)
    print(f"Agent ready with tools: {agent.tool_names}")

## 5. Test the Agent on a Current Events Question

The agent will automatically invoke WebSearch and return a grounded answer.

In [ ]:
QUERY = "What is today's news around the world?"

with mcp_client:
    tools = mcp_client.list_tools_sync()
    model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.7, max_tokens=1024)
    agent = Agent(model=model, tools=tools, system_prompt=SYSTEM_PROMPT)

    print(f"Query: {QUERY}")
    print("-" * 60)
    response = agent(QUERY)

print("\n" + "=" * 60)
print("AGENT RESPONSE")
print("=" * 60)
if hasattr(response, "message"):
    for block in response.message.get("content", []):
        if block.get("text"):
            print(block["text"])
else:
    print(str(response))

## 6. Try More Queries

The WebSearch tool handles queries up to 200 characters. Here are a few more examples showing different query types.

In [ ]:
demo_queries = [
    "What are the latest developments in quantum computing in 2026?",
    "What is the current price of gold?",
    "What new AWS services were announced recently?"
]

with mcp_client:
    tools = mcp_client.list_tools_sync()
    model = BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.7, max_tokens=1024)
    agent = Agent(model=model, tools=tools, system_prompt=SYSTEM_PROMPT)

    for query in demo_queries:
        print(f"\n{'=' * 60}")
        print(f"Query: {query}")
        print("=" * 60)
        response = agent(query)
        if hasattr(response, "message"):
            for block in response.message.get("content", []):
                if block.get("text"):
                    print(block["text"])

## Conclusion

In this notebook you:
- Connected a Strands agent to the AgentCore Gateway via MCP Streamable HTTP
- Discovered the WebSearch tool dynamically at runtime
- Got grounded, cited answers to real-time questions

**Next**: Run notebook `03-web-search-langchain-agent.ipynb` for the same integration using LangChain and LangGraph.